In [30]:

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import polars as pl

from scipy.stats import norm
from sklearn.linear_model import LinearRegression

PROJECT_ROOT = Path("../..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rest_of_season.config import (
    SEASON,
    CURRENT_WEEK,
    N_SIMULATIONS,
    RANDOM_SEED,
    CURRENT_OUTPUT_DIR
)

TARGET_WEEK = CURRENT_WEEK

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"

ROS_OUTPUT_DIR = CURRENT_OUTPUT_DIR
ROS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Season:", SEASON)
print("Projection week:", TARGET_WEEK)
print("ROS output:", ROS_OUTPUT_DIR)


Season: 2026
Projection week: 9
ROS output: C:\Users\efriedman\Desktop\NFL-Season-Projections\rest_of_season\outputs\week_09


In [31]:
remaining_games = pd.read_parquet(
    ROS_OUTPUT_DIR / "remaining_schedule.parquet"
)

team_snapshot = pd.read_parquet(
    ROS_OUTPUT_DIR / "team_snapshot.parquet"
)

scoring_ratings = pd.read_parquet(
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_ros_scoring_ratings.parquet"
)

injury_strength = pd.read_parquet(
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_injury_adjusted_team_strength.parquet"
)

print("Remaining games:", len(remaining_games))
print("Teams:", len(team_snapshot))
print("Scoring ratings:", len(scoring_ratings))

assert len(team_snapshot) == 32
assert len(scoring_ratings) == 32
assert remaining_games["game_id"].is_unique

Remaining games: 151
Teams: 32
Scoring ratings: 32


In [32]:
strength_lookup = injury_strength[
    [
        "team",
        "weekly_team_strength",
        "injury_adjusted_team_strength"
    ]
].copy()

assert strength_lookup["team"].is_unique
assert len(strength_lookup) == 32

home_strength = strength_lookup.rename(columns={
    "team": "home_team",
    "weekly_team_strength": "home_base_strength",
    "injury_adjusted_team_strength": "home_injury_strength"
})

away_strength = strength_lookup.rename(columns={
    "team": "away_team",
    "weekly_team_strength": "away_base_strength",
    "injury_adjusted_team_strength": "away_injury_strength"
})

games = (
    remaining_games
    .merge(
        home_strength,
        on="home_team",
        how="left",
        validate="many_to_one"
    )
    .merge(
        away_strength,
        on="away_team",
        how="left",
        validate="many_to_one"
    )
)


# Carry forward the current injury-adjusted ratings
# throughout the remaining regular season.

games["home_model_strength"] = games["home_injury_strength"]

games["away_model_strength"] = games["away_injury_strength"]


games["neutral_strength_diff"] = (
    games["home_model_strength"]
    - games["away_model_strength"]
)

assert games[
    ["home_model_strength", "away_model_strength"]
].notna().all().all()

In [33]:
schedule_clean = pl.read_parquet(
    PROCESSED_DIR / "schedule_clean.parquet"
)

historical_home = schedule_clean.filter(
    (pl.col("season") >= 2015)
    & (pl.col("season") <= 2025)
    & (pl.col("location") == "Home")
    & pl.col("result").is_not_null()
)

HOME_FIELD_ADVANTAGE = (
    historical_home
    .select(pl.col("result").mean())
    .item()
)

historical_rest = (
    schedule_clean
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & pl.col("result").is_not_null()
        & pl.col("home_rest").is_not_null()
        & pl.col("away_rest").is_not_null()
    )
    .select(["result", "home_rest", "away_rest", "location"])
    .to_pandas()
)

historical_rest["rest_diff"] = (
    historical_rest["home_rest"]
    - historical_rest["away_rest"]
).clip(-7, 7)

historical_rest["home_field"] = (
    historical_rest["location"] == "Home"
).astype(int)

rest_model = LinearRegression()

rest_model.fit(
    historical_rest[["home_field", "rest_diff"]],
    historical_rest["result"]
)

REST_POINT_VALUE = rest_model.coef_[1]

historical_margin_std = (
    schedule_clean
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & pl.col("result").is_not_null()
    )
    .select(pl.col("result").std())
    .item()
)

print("Home-field advantage:", round(HOME_FIELD_ADVANTAGE, 3))
print("Rest adjustment per day:", round(REST_POINT_VALUE, 3))
print("Historical margin SD:", round(historical_margin_std, 3))

Home-field advantage: 1.764
Rest adjustment per day: 0.181
Historical margin SD: 14.201


In [34]:
required_schedule_cols = [
    "location", "home_rest", "away_rest"
]

missing = [
    c for c in required_schedule_cols
    if c not in games.columns
]

if missing:
    raise KeyError(
        f"Remaining schedule missing columns: {missing}"
    )

if games[["home_rest", "away_rest"]].isna().any().any():
    raise ValueError(
        "Missing future rest-day values. "
        "We need to derive them from the schedule "
        "before generating projections."
    )

games["home_field_adjustment"] = np.where(
    games["location"] == "Home",
    HOME_FIELD_ADVANTAGE,
    0.0
)

games["rest_diff"] = (
    games["home_rest"] - games["away_rest"]
).clip(-7, 7)

games["rest_adjustment"] = (
    games["rest_diff"] * REST_POINT_VALUE
)

games["expected_home_margin"] = (
    games["neutral_strength_diff"]
    + games["home_field_adjustment"]
    + games["rest_adjustment"]
)

games["home_win_probability"] = norm.cdf(
    games["expected_home_margin"]
    / historical_margin_std
)

games["away_win_probability"] = (
    1 - games["home_win_probability"]
)

games["predicted_winner"] = np.where(
    games["home_win_probability"] >= 0.5,
    games["home_team"],
    games["away_team"]
)

games["predicted_win_probability"] = np.maximum(
    games["home_win_probability"],
    games["away_win_probability"]
)

In [35]:
home_scoring = scoring_ratings.rename(columns={
    "team": "home_team",
    "current_off_ppg": "home_current_off_ppg",
    "current_def_ppg": "home_current_def_ppg"
})

away_scoring = scoring_ratings.rename(columns={
    "team": "away_team",
    "current_off_ppg": "away_current_off_ppg",
    "current_def_ppg": "away_current_def_ppg"
})

games = (
    games
    .merge(
        home_scoring,
        on="home_team",
        how="left",
        validate="many_to_one"
    )
    .merge(
        away_scoring,
        on="away_team",
        how="left",
        validate="many_to_one"
    )
)

scoring_cols = [
    "home_current_off_ppg",
    "home_current_def_ppg",
    "away_current_off_ppg",
    "away_current_def_ppg"
]

assert games[scoring_cols].notna().all().all()

games["raw_home_score"] = (
    games["home_current_off_ppg"]
    + games["away_current_def_ppg"]
) / 2

games["raw_away_score"] = (
    games["away_current_off_ppg"]
    + games["home_current_def_ppg"]
) / 2

games["projected_total"] = (
    games["raw_home_score"]
    + games["raw_away_score"]
)

games["projected_home_score"] = (
    games["projected_total"]
    + games["expected_home_margin"]
) / 2

games["projected_away_score"] = (
    games["projected_total"]
    - games["expected_home_margin"]
) / 2

In [36]:

# Diagnose remaining-game counts by week

print("Target week:", TARGET_WEEK)
print("Total remaining games:", len(games))
print("Week column dtype:", games["week"].dtype)

print("\nGames by week:")
display(
    games.groupby("week")
    .size()
    .reset_index(name="game_count")
    .sort_values("week")
)

print("\nWeek 9 games:", len(games.loc[games["week"] == 9]))
print("Weeks 10-18:", len(games.loc[games["week"] > 9]))

print("\nWeek 9 matchups:")
display(
    games.loc[
        games["week"] == 9,
        ["week", "away_team", "home_team"]
    ]
)


Target week: 9
Total remaining games: 151
Week column dtype: int32

Games by week:


,week,game_count
0,9,15
1,10,14
2,11,13
3,12,16
4,13,14
5,14,15
6,15,16
7,16,16
8,17,16
9,18,16



Week 9 games: 15
Weeks 10-18: 136

Week 9 matchups:


,week,away_team,home_team
0,9,JAX,BAL
1,9,ARI,SEA
2,9,CIN,ATL
3,9,CLE,NO
4,9,DAL,IND
5,9,DEN,CAR
6,9,DET,MIA
7,9,GB,NE
8,9,HOU,LAC
9,9,LA,WAS


In [37]:

# Validate remaining-game projections before saving

assert len(games) == 151
assert games["game_id"].is_unique

required_cols = [
    "expected_home_margin",
    "home_win_probability",
    "projected_home_score",
    "projected_away_score"
]

assert games[required_cols].notna().all().all()

week9 = games.loc[games["week"] == TARGET_WEEK].copy()
future = games.loc[games["week"] > TARGET_WEEK].copy()

assert len(week9) == 15
assert len(future) == 136

# Week 9 must use injury-adjusted strengths
assert np.allclose(
    week9["home_model_strength"],
    week9["home_injury_strength"]
)

assert np.allclose(
    week9["away_model_strength"],
    week9["away_injury_strength"]
)

# Future weeks also use injury-adjusted strengths
assert np.allclose(
    future["home_model_strength"],
    future["home_injury_strength"]
)

assert np.allclose(
    future["away_model_strength"],
    future["away_injury_strength"]
)

print("Remaining-game projection validation passed.")
print("Total remaining games:", len(games))
print("Week 9 games:", len(week9))
print("Weeks 10-18 games:", len(future))

display(
    week9[
        [
            "away_team",
            "home_team",
            "expected_home_margin",
            "home_win_probability",
            "projected_away_score",
            "projected_home_score"
        ]
    ].round(3)
)


Remaining-game projection validation passed.
Total remaining games: 151
Week 9 games: 15
Weeks 10-18 games: 136


,away_team,home_team,expected_home_margin,home_win_probability,projected_away_score,projected_home_score
0,JAX,BAL,-3.463,0.404,24.839,21.376
1,ARI,SEA,10.142,0.762,18.293,28.434
2,CIN,ATL,-3.488,0.403,25.681,22.193
3,CLE,NO,4.658,0.629,19.878,24.536
4,DAL,IND,-0.321,0.491,25.232,24.911
5,DEN,CAR,-5.997,0.336,26.154,20.156
6,DET,MIA,-5.605,0.347,27.046,21.441
7,GB,NE,2.574,0.572,20.543,23.116
8,HOU,LAC,-4.575,0.374,24.073,19.498
9,LA,WAS,-4.137,0.385,25.545,21.408


In [38]:

# weekly_projection_path = (
#     WEEKLY_DATA_DIR
#     / f"week_{TARGET_WEEK:02d}_projections.parquet"
# )

# weekly_games = pd.read_parquet(weekly_projection_path)

# weekly_games = weekly_games.loc[
#     weekly_games["week"] == TARGET_WEEK
# ].copy()

# comparison = games.loc[
#     games["week"] == TARGET_WEEK,
#     [
#         "game_id",
#         "expected_home_margin",
#         "projected_total",
#         "home_win_probability"
#     ]
# ].merge(
#     weekly_games[
#         [
#             "game_id",
#             "expected_home_margin",
#             "projected_total",
#             "home_win_probability"
#         ]
#     ],
#     on="game_id",
#     suffixes=("_ros", "_weekly"),
#     validate="one_to_one"
# )

# assert len(comparison) == len(
#     games.loc[games["week"] == TARGET_WEEK]
# )

# for metric in [
#     "expected_home_margin",
#     "projected_total",
#     "home_win_probability"
# ]:
#     assert np.allclose(
#         comparison[f"{metric}_ros"],
#         comparison[f"{metric}_weekly"],
#         atol=1e-8
#     ), f"Mismatch in {metric}"

# print(
#     f"Week {TARGET_WEEK} projections match "
#     "the weekly model."
# )#


In [39]:
games["matchup"] = (
    games["away_team"] + " @ " + games["home_team"]
)

games["snapshot_week"] = TARGET_WEEK
games["season"] = SEASON

output_columns = [
    "season",
    "snapshot_week",
    "week",
    "game_id",
    "gameday",
    "away_team",
    "home_team",
    "matchup",
    "projected_away_score",
    "projected_home_score",
    "projected_total",
    "expected_home_margin",
    "away_win_probability",
    "home_win_probability",
    "predicted_winner",
    "predicted_win_probability"
]

ros_projections = games[
    output_columns
].sort_values(
    ["week", "gameday", "game_id"]
).reset_index(drop=True)

assert ros_projections["game_id"].is_unique
assert ros_projections["week"].ge(TARGET_WEEK).all()
assert ros_projections["week"].le(18).all()

ros_projections.to_parquet(
    ROS_OUTPUT_DIR / "ros_game_projections.parquet",
    index=False
)

ros_projections.to_csv(
    ROS_OUTPUT_DIR / "ros_game_projections.csv",
    index=False
)

print("Saved:", len(ros_projections), "remaining games")

display(ros_projections.head(20).round(3))

Saved: 151 remaining games


,season,snapshot_week,week,game_id,gameday,away_team,home_team,matchup,projected_away_score,projected_home_score,projected_total,expected_home_margin,away_win_probability,home_win_probability,predicted_winner,predicted_win_probability
0,2026,9,9,2026_09_JAX_BAL,2026-11-05,JAX,BAL,JAX @ BAL,24.839,21.376,46.215,-3.463,0.596,0.404,JAX,0.596
1,2026,9,9,2026_09_ARI_SEA,2026-11-08,ARI,SEA,ARI @ SEA,18.293,28.434,46.727,10.142,0.238,0.762,SEA,0.762
2,2026,9,9,2026_09_CIN_ATL,2026-11-08,CIN,ATL,CIN @ ATL,25.681,22.193,47.874,-3.488,0.597,0.403,CIN,0.597
3,2026,9,9,2026_09_CLE_NO,2026-11-08,CLE,NO,CLE @ NO,19.878,24.536,44.413,4.658,0.371,0.629,NO,0.629
4,2026,9,9,2026_09_DAL_IND,2026-11-08,DAL,IND,DAL @ IND,25.232,24.911,50.143,-0.321,0.509,0.491,DAL,0.509
5,2026,9,9,2026_09_DEN_CAR,2026-11-08,DEN,CAR,DEN @ CAR,26.154,20.156,46.310,-5.997,0.664,0.336,DEN,0.664
6,2026,9,9,2026_09_DET_MIA,2026-11-08,DET,MIA,DET @ MIA,27.046,21.441,48.487,-5.605,0.653,0.347,DET,0.653
7,2026,9,9,2026_09_GB_NE,2026-11-08,GB,NE,GB @ NE,20.543,23.116,43.659,2.574,0.428,0.572,NE,0.572
8,2026,9,9,2026_09_HOU_LAC,2026-11-08,HOU,LAC,HOU @ LAC,24.073,19.498,43.571,-4.575,0.626,0.374,HOU,0.626
9,2026,9,9,2026_09_LA_WAS,2026-11-08,LA,WAS,LA @ WAS,25.545,21.408,46.953,-4.137,0.615,0.385,LA,0.615
